In [1]:
%pip install scipy

  Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl (20.5 MB)

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv("Carothers_Grass_Morphometrics.csv", encoding="latin-1")
df.columns = [c.strip() for c in df.columns]

# Hacker et al. 2019, Table S4: shoots per plant (mean, 95% CI half-width, n)
hacker = {"AB": (2.7, 0.3, 180), "SP": (9.0, 1.5, 48), "UP": (1.6, 0.1, 407)}

rows = []
for sp, (h_mean, h_ci, h_n) in hacker.items():
    x = df.loc[df["Species"].str.strip() == sp, "Tillers per plant"].dropna()

    # recover Hacker's SD from the CI half-width
    h_sd = h_ci / stats.t.ppf(0.975, h_n - 1) * np.sqrt(h_n)

    t, p = stats.ttest_ind_from_stats(
        x.mean(), x.std(ddof=1), len(x),
        h_mean, h_sd, h_n,
        equal_var=False,           # Welch
    )
    rows.append(dict(species=sp, carothers_mean=x.mean(), n_c=len(x),
                     hacker_mean=h_mean, hacker_sd=h_sd, n_h=h_n, t=t, p=p))

print(pd.DataFrame(rows).round(4).to_string(index=False))

species  carothers_mean  n_c  hacker_mean  hacker_sd  n_h       t      p
     AB          3.6310   84          2.7     2.0397  180  3.9478 0.0001
     SP         32.2353   68          9.0     5.1658   48 16.9620 0.0000
     UP          3.4451   88          1.6     1.0262  407 11.6488 0.0000
